# MNIST DC-GAN — TensorFlow/Keras 변환 코드

이 노트북은 기존 PyTorch 기반 MNIST DC-GAN 코드를 **TensorFlow/Keras 코드**로 변환한 버전입니다.

목표는 MNIST 손글씨 숫자 이미지를 학습한 뒤, 생성자가 새로운 손글씨 숫자 이미지를 만들도록 학습하는 것입니다.

전체 구성은 다음과 같습니다.

1. 라이브러리 불러오기
2. 하이퍼파라미터 설정
3. MNIST 데이터 준비
4. 실제 이미지 확인
5. 생성자 모델 설계
6. 판별자 모델 설계
7. 가중치 초기화
8. 손실 함수와 최적화 알고리즘 설정
9. 생성 이미지 저장 및 출력 함수 작성
10. TensorFlow `GradientTape` 기반 GAN 학습 함수 작성
11. GAN 학습 실행
12. 손실 그래프 확인
13. 새 이미지 생성
14. 모델 저장과 불러오기
15. 성능 개선 실험 방향


## 1. 라이브러리 불러오기

아래 코드는 데이터 처리, 이미지 시각화, DCGAN 모델 생성, 학습, 저장에 필요한 라이브러리를 불러옵니다.


In [ ]:
# os는 폴더 생성, 파일 경로 처리, 기존 이미지 파일 삭제 등에 사용하는 표준 라이브러리입니다.
import os

# glob은 특정 패턴에 맞는 파일 목록을 찾을 때 사용하는 표준 라이브러리입니다.
# 예를 들어 dcgan_output 폴더 안의 PNG 파일을 한 번에 찾을 수 있습니다.
import glob

# time은 학습 시간을 측정하기 위해 사용하는 표준 라이브러리입니다.
from time import time

# numpy는 배열 기반 수치 계산을 처리하기 위해 사용하는 라이브러리입니다.
# 이미지 그리드 생성과 난수 고정 등에 사용합니다.
import numpy as np

# matplotlib.pyplot은 이미지와 손실 그래프를 화면에 출력하기 위해 사용합니다.
import matplotlib.pyplot as plt

# tensorflow는 딥러닝 모델 생성, 학습, 평가를 수행하는 대표적인 라이브러리입니다.
import tensorflow as tf

# keras는 TensorFlow 안에서 신경망 모델을 쉽게 구성할 수 있게 해주는 고수준 API입니다.
from tensorflow import keras

# layers는 Conv2D, Conv2DTranspose, BatchNormalization, Dense 같은 계층을 만들 때 사용합니다.
from tensorflow.keras import layers

# numpy 출력 옵션을 설정합니다.
# precision=3은 소수점 아래 3자리 정도만 출력한다는 뜻입니다.
np.set_printoptions(precision=3, suppress=True)

# TensorFlow 버전을 출력합니다.
print("TensorFlow version:", tf.__version__)

# TensorFlow가 인식하는 GPU 목록을 출력합니다.
# GPU가 있으면 DCGAN 학습 속도가 더 빨라질 수 있습니다.
print("사용 가능한 GPU:", tf.config.list_physical_devices("GPU"))


## 2. 하이퍼파라미터 설정

GAN은 생성자와 판별자가 서로 경쟁하면서 학습하는 구조입니다.

- 생성자: 무작위 노이즈에서 가짜 이미지를 생성합니다.
- 판별자: 입력 이미지가 진짜인지 가짜인지 판별합니다.

하이퍼파라미터는 학습 전에 사람이 직접 정하는 설정값입니다.


In [ ]:
# 실험 결과를 최대한 재현 가능하게 만들기 위해 난수 시드를 고정합니다.
SEED = 1234

# numpy 난수 시드를 고정합니다.
np.random.seed(SEED)

# TensorFlow 난수 시드를 고정합니다.
tf.random.set_seed(SEED)

# 전체 학습 반복 횟수입니다.
# 실습에서는 5 정도로 시작하고, 더 좋은 이미지를 얻으려면 20 이상으로 늘릴 수 있습니다.
EPOCHS = 5

# 한 번에 학습할 이미지 개수입니다.
# 너무 크면 메모리를 많이 사용하고, 너무 작으면 학습이 불안정할 수 있습니다.
BATCH_SIZE = 128

# 생성자에 입력할 노이즈 벡터의 차원입니다.
# 생성자는 이 100차원 노이즈를 바탕으로 28x28 이미지를 생성합니다.
NOISE_DIM = 100

# MNIST 이미지는 흑백 이미지이므로 채널 수는 1입니다.
IMAGE_CHANNELS = 1

# 생성하고 학습할 이미지의 가로, 세로 크기입니다.
# MNIST 이미지는 28x28 크기입니다.
IMAGE_SIZE = 28

# Adam 최적화 알고리즘의 학습률입니다.
# GAN은 학습이 불안정할 수 있으므로 일반 분류 모델보다 작은 학습률을 자주 사용합니다.
LEARNING_RATE = 0.0002

# Adam 최적화 알고리즘의 beta1 값입니다.
# DCGAN에서는 0.5를 자주 사용합니다.
BETA1 = 0.5

# Adam 최적화 알고리즘의 beta2 값입니다.
# 일반적으로 0.999를 많이 사용합니다.
BETA2 = 0.999

# 학습 결과 이미지를 저장할 폴더입니다.
OUTPUT_DIR = "dcgan_output_tf"

# 출력 폴더를 생성합니다.
# exist_ok=True는 이미 폴더가 있어도 오류를 내지 않도록 합니다.
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 기존 출력 이미지를 삭제하여 새 학습 결과만 남깁니다.
for file_path in glob.glob(os.path.join(OUTPUT_DIR, "*.png")):
    # 기존 PNG 파일을 삭제합니다.
    os.remove(file_path)

# 현재 설정값을 출력합니다.
print("SEED:", SEED)
print("EPOCHS:", EPOCHS)
print("BATCH_SIZE:", BATCH_SIZE)
print("NOISE_DIM:", NOISE_DIM)
print("IMAGE_CHANNELS:", IMAGE_CHANNELS)
print("IMAGE_SIZE:", IMAGE_SIZE)
print("LEARNING_RATE:", LEARNING_RATE)
print("BETA1:", BETA1)
print("BETA2:", BETA2)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 3. MNIST 데이터 준비

MNIST 이미지는 0부터 9까지의 손글씨 숫자 이미지입니다.

GAN에서는 숫자 라벨을 사용하지 않습니다.  
즉, 이미지가 0인지 1인지 맞히는 분류 문제가 아니라, **진짜 이미지 분포를 학습하여 가짜 이미지를 생성하는 문제**입니다.

이미지는 `[0, 255]` 범위에서 `[-1, 1]` 범위로 정규화합니다.  
생성자의 마지막 출력도 `tanh`를 사용하여 `[-1, 1]` 범위로 맞춥니다.


In [ ]:
# Keras에서 제공하는 MNIST 데이터셋을 불러옵니다.
# 반환값은 학습 데이터와 테스트 데이터로 나뉘지만, GAN 학습에서는 학습 이미지만 사용합니다.
(X_train, y_train), (_, _) = keras.datasets.mnist.load_data()

# 원본 학습 이미지 모양을 출력합니다.
# MNIST 원본 이미지는 (60000, 28, 28) 형태입니다.
print("원본 X_train shape:", X_train.shape)

# 원본 라벨 모양을 출력합니다.
# GAN에서는 라벨을 사용하지 않지만 데이터 확인용으로 출력합니다.
print("원본 y_train shape:", y_train.shape)

# 이미지를 float32 자료형으로 변환합니다.
# 딥러닝 모델은 일반적으로 float32 텐서를 사용합니다.
X_train = X_train.astype("float32")

# 원본 이미지는 0부터 255 사이의 픽셀값을 가집니다.
# 먼저 127.5를 빼고 다시 127.5로 나누면 값 범위가 [-1, 1]로 바뀝니다.
X_train = (X_train - 127.5) / 127.5

# Conv2D 계층은 입력을 (높이, 너비, 채널) 형태로 받습니다.
# MNIST는 흑백 이미지이므로 마지막 채널 차원 1을 추가합니다.
X_train = np.expand_dims(X_train, axis=-1)

# 정규화 후 이미지 모양을 출력합니다.
print("정규화 후 X_train shape:", X_train.shape)

# 정규화 후 최솟값을 출력합니다.
print("정규화 후 최솟값:", X_train.min())

# 정규화 후 최댓값을 출력합니다.
print("정규화 후 최댓값:", X_train.max())


In [ ]:
# tf.data.Dataset은 TensorFlow에서 데이터를 효율적으로 공급하기 위한 데이터 파이프라인입니다.
# GAN에서는 라벨을 사용하지 않으므로 이미지 X_train만 Dataset으로 만듭니다.
train_dataset = tf.data.Dataset.from_tensor_slices(X_train)

# shuffle은 학습 데이터 순서를 섞어 모델이 특정 순서에 의존하지 않게 합니다.
# buffer_size는 섞을 때 사용할 데이터 범위입니다.
train_dataset = train_dataset.shuffle(buffer_size=len(X_train), seed=SEED)

# batch는 이미지를 BATCH_SIZE개씩 묶어 미니배치를 만듭니다.
# drop_remainder=True는 마지막 배치 크기가 BATCH_SIZE보다 작으면 버립니다.
# GAN 학습에서는 배치 크기가 일정하면 라벨 생성이 편리합니다.
train_dataset = train_dataset.batch(BATCH_SIZE, drop_remainder=True)

# prefetch는 학습 중 다음 배치를 미리 준비하여 학습 속도를 높입니다.
# AUTOTUNE은 TensorFlow가 적절한 값을 자동으로 선택하도록 합니다.
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

# 데이터셋에서 첫 번째 배치를 하나 꺼냅니다.
sample_real_images = next(iter(train_dataset))

# 첫 번째 배치의 모양을 출력합니다.
print("배치 이미지 모양:", sample_real_images.shape)


## 4. 실제 이미지 확인

정규화된 이미지는 `[-1, 1]` 범위입니다.  
화면에 보기 위해서는 다시 `[0, 1]` 범위로 바꿔서 출력합니다.


In [ ]:
# [-1, 1] 범위 이미지를 [0, 1] 범위로 되돌리는 함수를 정의합니다.
def denormalize_images(images):
    # images가 TensorFlow 텐서일 수 있으므로 numpy 배열로 변환합니다.
    images = np.array(images)

    # [-1, 1] 범위를 [0, 1] 범위로 변환합니다.
    images = (images + 1.0) / 2.0

    # 값이 혹시 0보다 작거나 1보다 크면 0~1 범위로 잘라냅니다.
    images = np.clip(images, 0.0, 1.0)

    # 변환된 이미지를 반환합니다.
    return images

# 여러 이미지를 5x5 격자로 출력하는 함수를 정의합니다.
def show_image_grid(images, title="Image Grid", nrow=5):
    # 입력 이미지 중 앞에서 nrow*nrow개만 사용합니다.
    images = images[: nrow * nrow]

    # 이미지를 화면 출력용 [0, 1] 범위로 변환합니다.
    images = denormalize_images(images)

    # 그래프 크기를 설정합니다.
    plt.figure(figsize=(6, 6))

    # 이미지 개수만큼 반복합니다.
    for idx, image in enumerate(images):
        # 5x5 격자 중 idx+1번째 위치에 그림을 그립니다.
        plt.subplot(nrow, nrow, idx + 1)

        # 마지막 채널 차원을 제거하여 흑백 이미지로 표시합니다.
        plt.imshow(image.squeeze(), cmap="gray")

        # 각 작은 이미지의 축 눈금을 숨깁니다.
        plt.axis("off")

    # 전체 그림 제목을 표시합니다.
    plt.suptitle(title)

    # 그림 간격을 조정합니다.
    plt.tight_layout()

    # 이미지를 화면에 표시합니다.
    plt.show()

# 실제 MNIST 이미지 일부를 출력합니다.
show_image_grid(sample_real_images, title="Real MNIST Images")


## 5. 생성자 모델 설계

생성자는 무작위 노이즈 벡터를 입력받아 28×28 흑백 이미지를 만듭니다.

TensorFlow/Keras에서는 PyTorch의 `ConvTranspose2d`에 해당하는 계층으로 `Conv2DTranspose`를 사용합니다.

입력 모양은 다음과 같습니다.

`[배치크기, 100]`

생성자 내부 흐름은 다음과 같습니다.

`[B, 100] → [B, 1, 1, 100] → [B, 7, 7, 256] → [B, 14, 14, 128] → [B, 28, 28, 64] → [B, 28, 28, 1]`

마지막에는 `tanh`를 사용하여 출력 이미지 범위를 `[-1, 1]`로 맞춥니다.


In [ ]:
# 생성자 모델을 만드는 함수를 정의합니다.
def build_generator(noise_dim=100, image_channels=1):
    # 생성자 입력은 100차원 무작위 노이즈 벡터입니다.
    noise_input = layers.Input(shape=(noise_dim,))

    # Dense 계층은 100차원 노이즈를 7*7*256개의 숫자로 확장합니다.
    # 이것은 나중에 7x7 크기의 256채널 특징맵으로 바뀝니다.
    x = layers.Dense(
        units=7 * 7 * 256,
        use_bias=False
    )(noise_input)

    # BatchNormalization은 학습 중 특징값의 분포를 안정화합니다.
    x = layers.BatchNormalization()(x)

    # ReLU는 음수는 0으로, 양수는 그대로 통과시켜 비선형성을 추가합니다.
    x = layers.ReLU()(x)

    # 1차원 벡터를 7x7x256 특징맵으로 변환합니다.
    x = layers.Reshape((7, 7, 256))(x)

    # Conv2DTranspose는 작은 특징맵을 더 큰 특징맵으로 확대하는 계층입니다.
    # 여기서는 7x7x256을 14x14x128로 확대합니다.
    x = layers.Conv2DTranspose(
        filters=128,
        kernel_size=4,
        strides=2,
        padding="same",
        use_bias=False
    )(x)

    # 128개 특징맵의 분포를 안정화합니다.
    x = layers.BatchNormalization()(x)

    # 비선형성을 추가합니다.
    x = layers.ReLU()(x)

    # 14x14x128을 28x28x64로 확대합니다.
    x = layers.Conv2DTranspose(
        filters=64,
        kernel_size=4,
        strides=2,
        padding="same",
        use_bias=False
    )(x)

    # 64개 특징맵의 분포를 안정화합니다.
    x = layers.BatchNormalization()(x)

    # 비선형성을 추가합니다.
    x = layers.ReLU()(x)

    # 28x28x64를 최종 28x28x1 이미지로 변환합니다.
    # activation="tanh"는 출력 범위를 [-1, 1]로 맞춥니다.
    output_image = layers.Conv2D(
        filters=image_channels,
        kernel_size=3,
        strides=1,
        padding="same",
        activation="tanh",
        use_bias=False
    )(x)

    # 입력과 출력을 연결하여 Keras Model 객체를 생성합니다.
    model = keras.Model(noise_input, output_image, name="Generator")

    # 생성자 모델을 반환합니다.
    return model

# 생성자 모델 객체를 생성합니다.
generator = build_generator(NOISE_DIM, IMAGE_CHANNELS)

# 생성자 구조를 출력합니다.
generator.summary()


## 6. 판별자 모델 설계

판별자는 이미지가 진짜인지 가짜인지 판별합니다.

TensorFlow/Keras에서는 PyTorch의 `Conv2d`에 해당하는 계층으로 `Conv2D`를 사용합니다.

입력 모양은 다음과 같습니다.

`[배치크기, 28, 28, 1]`

출력 모양은 다음과 같습니다.

`[배치크기, 1]`

출력값은 확률이 아니라 `logit`입니다.  
따라서 마지막 계층에 `sigmoid`를 넣지 않습니다.

그 이유는 손실 함수인 `BinaryCrossentropy(from_logits=True)`가 내부적으로 sigmoid 계산을 함께 처리하기 때문입니다.


In [ ]:
# 판별자 모델을 만드는 함수를 정의합니다.
def build_discriminator(image_size=28, image_channels=1):
    # 판별자 입력은 28x28x1 이미지입니다.
    image_input = layers.Input(shape=(image_size, image_size, image_channels))

    # 첫 번째 Conv2D 계층은 28x28x1 이미지를 14x14x64 특징맵으로 압축합니다.
    x = layers.Conv2D(
        filters=64,
        kernel_size=4,
        strides=2,
        padding="same",
        use_bias=False
    )(image_input)

    # LeakyReLU는 음수 입력도 일부 통과시켜 기울기 소실을 줄입니다.
    # GAN의 판별자에서는 일반 ReLU보다 LeakyReLU를 자주 사용합니다.
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    # 두 번째 Conv2D 계층은 14x14x64를 7x7x128로 압축합니다.
    x = layers.Conv2D(
        filters=128,
        kernel_size=4,
        strides=2,
        padding="same",
        use_bias=False
    )(x)

    # BatchNormalization은 판별자 특징값의 분포를 안정화합니다.
    x = layers.BatchNormalization()(x)

    # LeakyReLU로 비선형성을 추가합니다.
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    # 세 번째 Conv2D 계층은 7x7x128을 더 깊은 특징맵으로 변환합니다.
    # strides=1과 padding="same"을 사용해 공간 크기는 7x7로 유지합니다.
    x = layers.Conv2D(
        filters=256,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False
    )(x)

    # 256개 특징맵의 분포를 안정화합니다.
    x = layers.BatchNormalization()(x)

    # LeakyReLU로 비선형성을 추가합니다.
    x = layers.LeakyReLU(negative_slope=0.2)(x)

    # 3차원 특징맵을 1차원 벡터로 펼칩니다.
    x = layers.Flatten()(x)

    # Dropout은 학습 중 일부 뉴런을 무작위로 꺼서 판별자의 과적합을 줄입니다.
    x = layers.Dropout(rate=0.3)(x)

    # 마지막 Dense 계층은 진짜/가짜 판별 점수 logit 1개를 출력합니다.
    # sigmoid를 넣지 않는 이유는 손실 함수에서 from_logits=True를 사용할 것이기 때문입니다.
    output_logit = layers.Dense(units=1)(x)

    # 입력과 출력을 연결하여 Keras Model 객체를 생성합니다.
    model = keras.Model(image_input, output_logit, name="Discriminator")

    # 판별자 모델을 반환합니다.
    return model

# 판별자 모델 객체를 생성합니다.
discriminator = build_discriminator(IMAGE_SIZE, IMAGE_CHANNELS)

# 판별자 구조를 출력합니다.
discriminator.summary()


## 7. 가중치 초기화와 모델 동작 확인

DCGAN에서는 합성곱 계층의 가중치를 평균 0, 표준편차 0.02 정도의 정규분포로 초기화하는 방식을 자주 사용합니다.

TensorFlow/Keras에서는 계층을 만들 때 `kernel_initializer`를 직접 넣을 수도 있습니다.  
여기서는 이미 모델을 만든 뒤 각 계층의 가중치를 직접 초기화하는 함수를 작성합니다.


In [ ]:
# DCGAN에서 자주 사용하는 가중치 초기화 함수를 정의합니다.
def initialize_dcgan_weights(model):
    # 모델 내부의 모든 계층을 하나씩 반복합니다.
    for layer in model.layers:
        # Conv2D, Conv2DTranspose, Dense 계층은 kernel 가중치를 가집니다.
        if isinstance(layer, (layers.Conv2D, layers.Conv2DTranspose, layers.Dense)):
            # 현재 계층의 가중치 목록을 가져옵니다.
            weights = layer.get_weights()

            # 가중치가 하나 이상 있으면 초기화를 수행합니다.
            if len(weights) > 0:
                # 첫 번째 원소는 kernel 가중치입니다.
                kernel_shape = weights[0].shape

                # 평균 0, 표준편차 0.02의 정규분포로 새 kernel을 생성합니다.
                new_kernel = np.random.normal(
                    loc=0.0,
                    scale=0.02,
                    size=kernel_shape
                ).astype("float32")

                # bias가 있는 계층이면 bias는 0으로 초기화합니다.
                if len(weights) == 2:
                    # 두 번째 원소는 bias입니다.
                    bias_shape = weights[1].shape

                    # bias를 0으로 초기화합니다.
                    new_bias = np.zeros(bias_shape, dtype="float32")

                    # 새 kernel과 새 bias를 계층에 설정합니다.
                    layer.set_weights([new_kernel, new_bias])
                else:
                    # bias가 없으면 kernel만 계층에 설정합니다.
                    layer.set_weights([new_kernel])

        # BatchNormalization 계층은 gamma와 beta를 가집니다.
        elif isinstance(layer, layers.BatchNormalization):
            # BatchNormalization의 가중치 목록을 가져옵니다.
            weights = layer.get_weights()

            # 일반적으로 [gamma, beta, moving_mean, moving_variance] 구조입니다.
            if len(weights) == 4:
                # gamma는 평균 1, 표준편차 0.02의 정규분포로 초기화합니다.
                gamma = np.random.normal(
                    loc=1.0,
                    scale=0.02,
                    size=weights[0].shape
                ).astype("float32")

                # beta는 0으로 초기화합니다.
                beta = np.zeros(weights[1].shape, dtype="float32")

                # moving_mean은 0으로 초기화합니다.
                moving_mean = np.zeros(weights[2].shape, dtype="float32")

                # moving_variance는 1로 초기화합니다.
                moving_variance = np.ones(weights[3].shape, dtype="float32")

                # 초기화된 값을 BatchNormalization 계층에 설정합니다.
                layer.set_weights([gamma, beta, moving_mean, moving_variance])

# 생성자 가중치를 초기화합니다.
initialize_dcgan_weights(generator)

# 판별자 가중치를 초기화합니다.
initialize_dcgan_weights(discriminator)

# 더미 노이즈를 생성합니다.
# TensorFlow의 생성자 입력은 [배치크기, NOISE_DIM] 형태입니다.
dummy_noise = tf.random.normal(shape=(4, NOISE_DIM))

# 더미 노이즈를 생성자에 넣어 가짜 이미지를 생성합니다.
dummy_fake = generator(dummy_noise, training=False)

# 생성된 가짜 이미지를 판별자에 넣어 진짜/가짜 점수 logit을 계산합니다.
dummy_score = discriminator(dummy_fake, training=False)

# 생성자 출력 모양을 출력합니다.
print("생성자 출력 모양:", dummy_fake.shape)

# 판별자 출력 모양을 출력합니다.
print("판별자 출력 모양:", dummy_score.shape)


## 8. 손실 함수와 최적화 알고리즘 설정

### 8.1 손실 함수: Binary Crossentropy

판별자는 입력 이미지가 진짜인지 가짜인지 판별하는 이진 분류 문제를 풉니다.  
따라서 이진 분류 손실 함수인 `BinaryCrossentropy`를 사용합니다.

여기서는 판별자의 마지막 출력이 확률이 아니라 logit이므로 다음과 같이 설정합니다.

`BinaryCrossentropy(from_logits=True)`

### 8.2 판별자 손실

판별자는 두 가지를 잘해야 합니다.

1. 진짜 이미지를 진짜라고 판단해야 합니다.
2. 가짜 이미지를 가짜라고 판단해야 합니다.

따라서 판별자 손실은 다음 두 손실의 합입니다.

`진짜 이미지 손실 + 가짜 이미지 손실`

### 8.3 생성자 손실

생성자는 판별자를 속이는 것이 목표입니다.  
즉, 생성자가 만든 가짜 이미지를 판별자가 진짜라고 판단하도록 학습합니다.

### 8.4 최적화 알고리즘: Adam

DCGAN에서는 다음 설정의 Adam을 자주 사용합니다.

- 학습률: `0.0002`
- beta1: `0.5`
- beta2: `0.999`


In [ ]:
# BinaryCrossentropy 손실 함수를 생성합니다.
# from_logits=True는 판별자 출력이 sigmoid 확률이 아니라 logit이라는 뜻입니다.
cross_entropy = keras.losses.BinaryCrossentropy(from_logits=True)

# 생성자 전용 Adam 최적화 알고리즘을 생성합니다.
generator_optimizer = keras.optimizers.Adam(
    learning_rate=LEARNING_RATE,
    beta_1=BETA1,
    beta_2=BETA2
)

# 판별자 전용 Adam 최적화 알고리즘을 생성합니다.
discriminator_optimizer = keras.optimizers.Adam(
    learning_rate=LEARNING_RATE,
    beta_1=BETA1,
    beta_2=BETA2
)

# 생성 이미지 변화를 비교하기 위해 고정 노이즈를 생성합니다.
# 학습 중 같은 노이즈를 계속 넣어야 이미지가 어떻게 발전하는지 비교할 수 있습니다.
fixed_noise = tf.random.normal(shape=(25, NOISE_DIM), seed=SEED)

# 설정된 손실 함수와 최적화 알고리즘을 출력합니다.
print("손실 함수:", cross_entropy)
print("생성자 최적화 알고리즘:", generator_optimizer)
print("판별자 최적화 알고리즘:", discriminator_optimizer)


In [ ]:
# 판별자 손실을 계산하는 함수를 정의합니다.
def discriminator_loss(real_logits, fake_logits):
    # 진짜 이미지에 대한 정답 라벨은 1입니다.
    # real_logits와 같은 모양의 1 텐서를 만듭니다.
    real_labels = tf.ones_like(real_logits)

    # 가짜 이미지에 대한 정답 라벨은 0입니다.
    # fake_logits와 같은 모양의 0 텐서를 만듭니다.
    fake_labels = tf.zeros_like(fake_logits)

    # 진짜 이미지를 진짜라고 맞히는 손실을 계산합니다.
    real_loss = cross_entropy(real_labels, real_logits)

    # 가짜 이미지를 가짜라고 맞히는 손실을 계산합니다.
    fake_loss = cross_entropy(fake_labels, fake_logits)

    # 판별자 전체 손실은 진짜 손실과 가짜 손실의 합입니다.
    total_loss = real_loss + fake_loss

    # 전체 판별자 손실을 반환합니다.
    return total_loss

# 생성자 손실을 계산하는 함수를 정의합니다.
def generator_loss(fake_logits):
    # 생성자는 가짜 이미지를 진짜처럼 보이게 만들어야 합니다.
    # 따라서 가짜 이미지에 대한 목표 라벨을 1로 둡니다.
    target_labels = tf.ones_like(fake_logits)

    # 판별자가 가짜 이미지를 진짜로 판단하도록 만드는 손실을 계산합니다.
    loss = cross_entropy(target_labels, fake_logits)

    # 생성자 손실을 반환합니다.
    return loss


## 9. 생성 이미지 저장 및 출력 함수 작성

학습이 진행되면서 생성자가 만드는 이미지가 어떻게 변하는지 확인하기 위해, 고정 노이즈로 생성한 이미지를 저장하고 출력합니다.


In [ ]:
# 생성 이미지를 PNG 파일로 저장하는 함수를 정의합니다.
def save_generated_images(epoch, generator, fixed_noise, output_dir):
    # 생성자를 사용하여 고정 노이즈에서 가짜 이미지를 생성합니다.
    # training=False는 BatchNormalization을 추론 모드로 사용한다는 뜻입니다.
    fake_images = generator(fixed_noise, training=False)

    # 화면 출력과 저장을 위해 [0, 1] 범위로 변환합니다.
    images = denormalize_images(fake_images)

    # 그래프 크기를 설정합니다.
    plt.figure(figsize=(6, 6))

    # 25개 이미지를 5x5 격자로 표시합니다.
    for idx in range(25):
        # 5x5 격자 중 idx+1번째 위치를 선택합니다.
        plt.subplot(5, 5, idx + 1)

        # 흑백 이미지를 출력합니다.
        plt.imshow(images[idx].squeeze(), cmap="gray")

        # 축 눈금을 숨깁니다.
        plt.axis("off")

    # 전체 그림 제목을 설정합니다.
    plt.suptitle(f"Generated Images - Epoch {epoch}")

    # 저장 경로를 생성합니다.
    save_path = os.path.join(output_dir, f"epoch_{epoch:03d}.png")

    # 이미지 간격을 조정합니다.
    plt.tight_layout()

    # 그림을 PNG 파일로 저장합니다.
    plt.savefig(save_path)

    # 화면에 이미지를 출력합니다.
    plt.show()

    # 저장 경로를 반환합니다.
    return save_path


## 10. TensorFlow `GradientTape` 기반 GAN 학습 함수 작성

일반적인 Keras 모델은 `model.fit()`으로 쉽게 학습할 수 있습니다.  
하지만 GAN은 생성자와 판별자를 번갈아 학습해야 하므로 직접 학습 루프를 작성하는 것이 이해하기 쉽습니다.

TensorFlow에서는 `tf.GradientTape`를 사용하여 다음 과정을 직접 구현합니다.

1. 실제 이미지를 판별자에 넣어 진짜 점수를 계산합니다.
2. 생성자가 무작위 노이즈에서 가짜 이미지를 만듭니다.
3. 가짜 이미지를 판별자에 넣어 가짜 점수를 계산합니다.
4. 판별자 손실과 생성자 손실을 계산합니다.
5. 각 손실에 대한 기울기를 계산합니다.
6. 각 최적화 알고리즘으로 생성자와 판별자의 가중치를 업데이트합니다.


In [ ]:
# 하나의 미니배치에 대해 GAN 학습 1단계를 수행하는 함수를 정의합니다.
# @tf.function은 Python 함수를 TensorFlow 그래프 함수로 변환하여 실행 속도를 높일 수 있습니다.
@tf.function
def train_step(real_images):
    # 현재 미니배치 크기를 계산합니다.
    current_batch_size = tf.shape(real_images)[0]

    # 생성자 입력으로 사용할 무작위 노이즈를 생성합니다.
    noise = tf.random.normal(shape=(current_batch_size, NOISE_DIM))

    # GradientTape는 연산 과정을 기록하여 나중에 기울기를 계산할 수 있게 합니다.
    # 생성자와 판별자를 각각 업데이트해야 하므로 tape를 두 개 사용합니다.
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # 생성자가 무작위 노이즈에서 가짜 이미지를 생성합니다.
        # training=True는 BatchNormalization을 학습 모드로 사용한다는 뜻입니다.
        generated_images = generator(noise, training=True)

        # 판별자가 실제 이미지에 대해 진짜/가짜 점수 logit을 출력합니다.
        real_logits = discriminator(real_images, training=True)

        # 판별자가 생성 이미지에 대해 진짜/가짜 점수 logit을 출력합니다.
        fake_logits = discriminator(generated_images, training=True)

        # 생성자 손실을 계산합니다.
        gen_loss = generator_loss(fake_logits)

        # 판별자 손실을 계산합니다.
        disc_loss = discriminator_loss(real_logits, fake_logits)

    # 생성자 손실에 대한 생성자 파라미터의 기울기를 계산합니다.
    gradients_of_generator = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    # 판별자 손실에 대한 판별자 파라미터의 기울기를 계산합니다.
    gradients_of_discriminator = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    # 계산된 기울기를 사용하여 생성자 가중치를 업데이트합니다.
    generator_optimizer.apply_gradients(
        zip(gradients_of_generator, generator.trainable_variables)
    )

    # 계산된 기울기를 사용하여 판별자 가중치를 업데이트합니다.
    discriminator_optimizer.apply_gradients(
        zip(gradients_of_discriminator, discriminator.trainable_variables)
    )

    # 현재 미니배치의 생성자 손실과 판별자 손실을 반환합니다.
    return gen_loss, disc_loss


In [ ]:
# GAN 전체 학습 함수를 정의합니다.
def train_gan(train_dataset, epochs):
    # 학습 시작 시간을 기록합니다.
    start_time = time()

    # 에포크별 생성자 손실을 저장할 리스트입니다.
    g_losses = []

    # 에포크별 판별자 손실을 저장할 리스트입니다.
    d_losses = []

    # 지정한 에포크 수만큼 반복합니다.
    for epoch in range(1, epochs + 1):
        # 현재 에포크의 생성자 손실 평균 계산을 위한 객체입니다.
        epoch_g_loss = keras.metrics.Mean()

        # 현재 에포크의 판별자 손실 평균 계산을 위한 객체입니다.
        epoch_d_loss = keras.metrics.Mean()

        # 데이터셋에서 실제 이미지를 미니배치 단위로 가져옵니다.
        for batch_idx, real_images in enumerate(train_dataset):
            # 현재 배치로 GAN 학습 1단계를 수행합니다.
            gen_loss, disc_loss = train_step(real_images)

            # 생성자 손실 평균 객체에 현재 배치 손실을 추가합니다.
            epoch_g_loss.update_state(gen_loss)

            # 판별자 손실 평균 객체에 현재 배치 손실을 추가합니다.
            epoch_d_loss.update_state(disc_loss)

        # 현재 에포크의 평균 생성자 손실을 숫자로 가져옵니다.
        avg_g_loss = float(epoch_g_loss.result().numpy())

        # 현재 에포크의 평균 판별자 손실을 숫자로 가져옵니다.
        avg_d_loss = float(epoch_d_loss.result().numpy())

        # 평균 생성자 손실을 리스트에 저장합니다.
        g_losses.append(avg_g_loss)

        # 평균 판별자 손실을 리스트에 저장합니다.
        d_losses.append(avg_d_loss)

        # 현재 에포크의 학습 결과를 출력합니다.
        print(
            f"Epoch [{epoch}/{epochs}] "
            f"Generator Loss: {avg_g_loss:.4f} "
            f"Discriminator Loss: {avg_d_loss:.4f}"
        )

        # 현재 에포크의 생성 이미지를 저장하고 출력합니다.
        save_path = save_generated_images(
            epoch=epoch,
            generator=generator,
            fixed_noise=fixed_noise,
            output_dir=OUTPUT_DIR
        )

        # 이미지 저장 경로를 출력합니다.
        print("생성 이미지 저장:", save_path)

    # 전체 학습 시간을 계산합니다.
    total_time = time() - start_time

    # 전체 학습 시간을 출력합니다.
    print(f"전체 학습 시간: {total_time:.2f}초")

    # 생성자 손실 목록과 판별자 손실 목록을 반환합니다.
    return g_losses, d_losses


## 11. GAN 학습 실행

GAN 학습은 일반 분류 모델보다 불안정할 수 있습니다.  
손실값이 단순히 계속 감소하지 않을 수 있으며, 생성 이미지 품질을 함께 확인해야 합니다.

처음 몇 epoch에서는 이미지가 노이즈처럼 보일 수 있습니다.  
학습 epoch를 늘리면 점차 숫자와 비슷한 형태가 나타납니다.


In [ ]:
# GAN 학습을 실행합니다.
# 반환값은 에포크별 생성자 손실과 판별자 손실입니다.
g_losses, d_losses = train_gan(
    train_dataset=train_dataset,
    epochs=EPOCHS
)


## 12. 손실 그래프 확인

GAN에서는 생성자 손실과 판별자 손실이 서로 경쟁적으로 변합니다.  
두 손실 중 하나만 계속 낮아지는 것이 항상 좋은 것은 아닙니다.

생성 이미지가 점점 숫자처럼 보이는지도 함께 확인해야 합니다.


In [ ]:
# 에포크 번호를 생성합니다.
epochs_range = range(1, EPOCHS + 1)

# 손실 그래프를 그릴 그림을 생성합니다.
plt.figure(figsize=(8, 5))

# 생성자 손실 그래프를 그립니다.
plt.plot(epochs_range, g_losses, marker="o", label="Generator Loss")

# 판별자 손실 그래프를 그립니다.
plt.plot(epochs_range, d_losses, marker="o", label="Discriminator Loss")

# 그래프 제목을 지정합니다.
plt.title("DC-GAN Loss Curve")

# x축 이름을 지정합니다.
plt.xlabel("Epoch")

# y축 이름을 지정합니다.
plt.ylabel("Loss")

# 범례를 표시합니다.
plt.legend()

# 격자를 표시합니다.
plt.grid(True)

# 그래프를 출력합니다.
plt.show()


## 13. 새 이미지 생성

학습된 생성자에 새로운 무작위 노이즈를 넣어 숫자 이미지를 생성합니다.


In [ ]:
# 생성할 이미지 개수를 지정합니다.
num_generate = 25

# 새로운 무작위 노이즈를 생성합니다.
new_noise = tf.random.normal(shape=(num_generate, NOISE_DIM))

# 생성자를 사용하여 새로운 가짜 이미지를 생성합니다.
# training=False는 추론 모드로 이미지를 생성한다는 뜻입니다.
generated_images = generator(new_noise, training=False)

# 생성된 이미지를 화면에 출력합니다.
show_image_grid(generated_images, title="New Generated Images")


## 14. 모델 저장과 불러오기

GAN은 생성자와 판별자 모델을 각각 저장할 수 있습니다.  
실제로 이미지를 생성할 때는 주로 생성자 모델을 사용합니다.

TensorFlow/Keras에서는 `.keras` 형식으로 전체 모델을 저장할 수 있습니다.


In [ ]:
# 생성자 모델을 저장할 파일 이름입니다.
GENERATOR_PATH = "dcgan_generator_tf.keras"

# 판별자 모델을 저장할 파일 이름입니다.
DISCRIMINATOR_PATH = "dcgan_discriminator_tf.keras"

# 생성자 모델 전체를 저장합니다.
# 모델 구조와 학습된 가중치가 함께 저장됩니다.
generator.save(GENERATOR_PATH)

# 판별자 모델 전체를 저장합니다.
# 모델 구조와 학습된 가중치가 함께 저장됩니다.
discriminator.save(DISCRIMINATOR_PATH)

# 저장 완료 메시지를 출력합니다.
print("생성자 저장 완료:", GENERATOR_PATH)

# 저장 완료 메시지를 출력합니다.
print("판별자 저장 완료:", DISCRIMINATOR_PATH)

# 저장된 생성자 모델을 다시 불러옵니다.
loaded_generator = keras.models.load_model(GENERATOR_PATH)

# 불러오기 완료 메시지를 출력합니다.
print("생성자 모델 불러오기 완료")

# 불러온 생성자에 새 노이즈를 넣어 이미지 생성이 가능한지 확인합니다.
loaded_generated_images = loaded_generator(new_noise, training=False)

# 불러온 모델이 만든 이미지를 출력합니다.
show_image_grid(loaded_generated_images, title="Loaded Generator Images")


## 15. PyTorch 코드에서 TensorFlow/Keras 코드로 바뀐 핵심 차이

기존 PyTorch 코드와 비교했을 때 핵심 변경점은 다음과 같습니다.

1. `torchvision.datasets.MNIST` 대신 `keras.datasets.mnist.load_data()`를 사용했습니다.
2. PyTorch의 이미지 형식 `[배치, 채널, 높이, 너비]` 대신 TensorFlow 형식 `[배치, 높이, 너비, 채널]`을 사용했습니다.
3. `nn.ConvTranspose2d` 대신 `layers.Conv2DTranspose`를 사용했습니다.
4. `nn.Conv2d` 대신 `layers.Conv2D`를 사용했습니다.
5. `nn.BatchNorm2d` 대신 `layers.BatchNormalization`을 사용했습니다.
6. `nn.BCEWithLogitsLoss` 대신 `keras.losses.BinaryCrossentropy(from_logits=True)`를 사용했습니다.
7. PyTorch의 `loss.backward()` 대신 TensorFlow의 `tf.GradientTape()`를 사용했습니다.
8. PyTorch의 `optimizer.step()` 대신 TensorFlow의 `optimizer.apply_gradients()`를 사용했습니다.
9. `torch.save()` 대신 `model.save()`를 사용했습니다.

GAN은 생성자와 판별자를 따로 업데이트해야 하므로 TensorFlow/Keras에서도 `model.fit()`보다 직접 학습 루프를 작성하는 방식이 이해하기 쉽습니다.


## 16. 성능 개선 실험 방향

생성 이미지 품질을 높이려면 다음 항목을 실험할 수 있습니다.

1. `EPOCHS`를 20 이상으로 늘립니다.
2. `BATCH_SIZE`를 64 또는 128로 조정합니다.
3. 생성자와 판별자의 채널 수를 늘립니다.
4. 학습률을 `0.0001` 또는 `0.0002` 범위에서 조정합니다.
5. 생성자와 판별자의 학습 균형을 확인합니다.
6. 판별자가 너무 강하면 생성자가 학습하지 못할 수 있습니다.
7. 생성자가 너무 강하면 판별자가 진짜/가짜를 구분하지 못할 수 있습니다.
8. `label smoothing`을 적용하여 판별자가 지나치게 확신하지 않도록 만들 수 있습니다.
9. `LeakyReLU`, `BatchNormalization`, `Dropout` 위치를 조정해 볼 수 있습니다.
10. Fashion-MNIST나 CIFAR-10 같은 다른 이미지 데이터셋으로 확장해 볼 수 있습니다.
